# LeetCode 2085 — Count Common Words With One Occurrence bằng Python

**Link đề:** https://leetcode.com/problems/count-common-words-with-one-occurrence/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String, Counting

```python
class Solution:
    def countWords(self, words1: list[str], words2: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu chính xác điều kiện "xuất hiện **đúng một lần** trong **mỗi** mảng".
2. Đi từ cách đếm bằng `list.count` (`O(n·m)`) → **dict đếm tần suất** (`O(n + m)`) → `collections.Counter`.
3. Nắm cú pháp: `dict`, `dict.get(key, 0)`, `in`, `Counter`, `set`, phép giao `&`, generator trong `sum(...)`.
4. Trace từng bước hai bảng tần suất, chứng minh tính đúng, phân tích Time/Space.
5. Luyện các biến thể: "đúng k lần", "xuất hiện ở cả hai mảng bất kể số lần", ba mảng.

## 1. Đề bài và cách hiểu đúng

Cho hai mảng chuỗi `words1` và `words2`. Đếm số chuỗi **xuất hiện đúng 1 lần trong `words1`** **và** **đúng 1 lần trong `words2`**.

| `words1` | `words2` | Kết quả | Giải thích |
|---|---|:---:|---|
| `["leetcode","is","amazing","as","is"]` | `["amazing","leetcode","is"]` | `2` | `"leetcode"`, `"amazing"` đạt; `"is"` có 2 lần ở words1; `"as"` không có ở words2 |
| `["b","bb","bbb"]` | `["a","aa","aaa"]` | `0` | không có chuỗi chung |
| `["a","ab"]` | `["a","a","a","ab"]` | `1` | `"a"` có 3 lần ở words2 → loại; chỉ `"ab"` đạt |

**Constraints:** `1 <= len(words1), len(words2) <= 1000`; mỗi chuỗi dài 1..30, chỉ chữ thường.

Câu hỏi then chốt: *với mỗi từ, ta cần biết nó xuất hiện **bao nhiêu lần** ở mỗi mảng.* Đây là bài **đếm tần suất** (frequency counting).

In [1]:
# Ba ví dụ của đề, dùng lại nhiều lần trong notebook
examples = [
    (["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"], 2),
    (["b", "bb", "bbb"], ["a", "aa", "aaa"], 0),
    (["a", "ab"], ["a", "a", "a", "ab"], 1),
]
for w1, w2, expected in examples:
    print(w1, "|", w2, "-> kỳ vọng", expected)

['leetcode', 'is', 'amazing', 'as', 'is'] | ['amazing', 'leetcode', 'is'] -> kỳ vọng 2
['b', 'bb', 'bbb'] | ['a', 'aa', 'aaa'] -> kỳ vọng 0
['a', 'ab'] | ['a', 'a', 'a', 'ab'] -> kỳ vọng 1


### Đọc kỹ chữ "exactly once in each"

Điều kiện có **hai vế**, nối bằng **VÀ**:

- vế 1: số lần trong `words1` **bằng 1**;
- vế 2: số lần trong `words2` **bằng 1**.

Nghĩa là các trường hợp sau đều **không** được đếm:

| Lần trong words1 | Lần trong words2 | Đếm? |
|:---:|:---:|:---:|
| 1 | 1 | ✅ |
| 2 | 1 | ❌ |
| 1 | 0 | ❌ |
| 0 | 1 | ❌ |
| 3 | 3 | ❌ |

## 2. Nền tảng Python cần dùng

### 2.1. `list.count(x)` — đếm số lần `x` xuất hiện trong list

Đây là cách đếm dễ nhất, nhưng mỗi lần gọi phải **duyệt hết list** → tốn `O(n)`.

In [2]:
words = ["is", "a", "is", "b"]
print(words.count("is"))   # 2
print(words.count("a"))    # 1
print(words.count("zzz"))  # 0  (không có thì trả 0, không báo lỗi)

2
1
0


### 2.2. `dict` — bảng tra cứu key → value

Một `dict` lưu cặp **key: value**. Ở bài này: key là **từ**, value là **số lần xuất hiện**.

- Tạo rỗng: `freq = {}`
- Gán/cập nhật: `freq["is"] = 2`
- Đọc: `freq["is"]` (key không tồn tại → `KeyError`)
- Kiểm tra tồn tại: `"is" in freq` → trung bình `O(1)`

In [3]:
freq = {}
freq["is"] = 1
freq["is"] = freq["is"] + 1
freq["a"] = 1
print(freq)            # {'is': 2, 'a': 1}
print("is" in freq)    # True
print("zz" in freq)    # False
try:
    print(freq["zz"])
except KeyError as e:
    print("KeyError khi đọc key không có:", e)

{'is': 2, 'a': 1}
True
False
KeyError khi đọc key không có: 'zz'


### 2.3. `dict.get(key, default)` — đọc an toàn

`freq.get(w, 0)` trả `freq[w]` nếu có, ngược lại trả `0`. Nhờ vậy câu lệnh đếm gọn thành một dòng:

```python
freq[w] = freq.get(w, 0) + 1
```

In [4]:
freq = {}
for w in ["is", "a", "is"]:
    before = freq.get(w, 0)
    freq[w] = before + 1
    print(f"w={w!r}: trước={before}, sau={freq[w]}, freq={freq}")

w='is': trước=0, sau=1, freq={'is': 1}
w='a': trước=0, sau=1, freq={'is': 1, 'a': 1}
w='is': trước=1, sau=2, freq={'is': 2, 'a': 1}


### 2.4. `collections.Counter` — dict chuyên để đếm

`Counter(iterable)` đếm sẵn mọi phần tử. Điểm hay: đọc key **không có** sẽ trả `0` (không lỗi).

In [5]:
from collections import Counter

c = Counter(["leetcode", "is", "amazing", "as", "is"])
print(c)                 # Counter({'is': 2, 'leetcode': 1, ...})
print(c["is"], c["as"])  # 2 1
print(c["không_có"])     # 0  -> khác dict thường
print(c.items())

Counter({'is': 2, 'leetcode': 1, 'amazing': 1, 'as': 1})
2 1
0
dict_items([('leetcode', 1), ('is', 2), ('amazing', 1), ('as', 1)])


### 2.5. Duyệt `dict.items()` và đếm bằng `sum(1 for ...)`

- `for word, cnt in freq.items():` lấy đồng thời key và value.
- `sum(1 for x in data if điều_kiện)` đếm số phần tử thỏa điều kiện. Biểu thức `True`/`False` cũng có thể cộng trực tiếp vì `True == 1`.

In [6]:
freq = {"leetcode": 1, "is": 2, "amazing": 1, "as": 1}
for word, cnt in freq.items():
    print(word, "->", cnt)

print("Số từ xuất hiện đúng 1 lần:", sum(1 for cnt in freq.values() if cnt == 1))
print("True + True + False =", True + True + False)

leetcode -> 1
is -> 2
amazing -> 1
as -> 1
Số từ xuất hiện đúng 1 lần: 3
True + True + False = 2


### 2.6. `set` và phép giao `&`

`set` là tập hợp không trùng lặp. `A & B` là các phần tử có ở **cả** A và B. Ta sẽ dùng nó trong một lời giải khác.

In [7]:
A = {"leetcode", "amazing", "as"}
B = {"amazing", "leetcode", "is"}
print(A & B)   # {'leetcode', 'amazing'}  (thứ tự in có thể khác)
print(sorted(A & B))

{'leetcode', 'amazing'}
['amazing', 'leetcode']


## 3. Cách 1 — Trực giác: thử từng từ, đếm bằng `list.count`

Gợi ý của đề: *"Could you try every word?"*

Ý tưởng: duyệt từng từ `w` **khác nhau** trong `words1`; nếu `words1.count(w) == 1` và `words2.count(w) == 1` thì cộng 1.

Chú ý: phải duyệt các từ **khác nhau** (ví dụ dùng `set(words1)`), nếu không một từ lặp sẽ bị xét nhiều lần. Nhưng với điều kiện "đúng 1 lần" thì từ lặp vốn không được đếm, nên duyệt thẳng `words1` vẫn ra đúng — một quan sát nhỏ ta sẽ kiểm chứng.

In [8]:
def count_words_brute(words1: list[str], words2: list[str]) -> int:
    total = 0
    for w in set(words1):                 # mỗi từ khác nhau xét 1 lần
        if words1.count(w) == 1 and words2.count(w) == 1:
            total += 1
    return total

for w1, w2, expected in examples:
    print(count_words_brute(w1, w2), "kỳ vọng", expected)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


In [9]:
# Kiểm chứng: duyệt thẳng words1 (không set) vẫn đúng với điều kiện "đúng 1 lần"
def count_words_brute_noset(words1, words2):
    return sum(1 for w in words1 if words1.count(w) == 1 and words2.count(w) == 1)

for w1, w2, expected in examples:
    print(count_words_brute_noset(w1, w2), "kỳ vọng", expected)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


## 4. Nút thắt của cách trực giác

Gọi `n = len(words1)`, `m = len(words2)`.

- Với mỗi từ ta gọi `words1.count(w)` (`O(n)`) và `words2.count(w)` (`O(m)`).
- Có tới `n` từ → tổng `O(n·(n + m))`, mỗi phép so sánh chuỗi lại tốn tới 30 ký tự.

Với `n = m = 1000` là ~2 triệu phép so sánh — vẫn chạy được ở đây, nhưng nếu đề tăng lên `10^5` thì thành `2·10^10` → quá chậm.

**Công việc bị lặp:** ta đếm đi đếm lại cùng một mảng nhiều lần. Đếm **một lần cho tất cả từ** rồi tra cứu sẽ nhanh hơn nhiều.

In [10]:
# Đếm số phép so sánh mà cách brute phải làm (mô phỏng)
def brute_cost(n, m):
    return n * (n + m)

for n in [10, 1000, 100_000]:
    print(f"n=m={n:>7}: khoảng {brute_cost(n, n):,} phép so sánh")

n=m=     10: khoảng 200 phép so sánh
n=m=   1000: khoảng 2,000,000 phép so sánh
n=m= 100000: khoảng 20,000,000,000 phép so sánh


## 5. Bước đột phá — Đếm một lần, tra nhiều lần

Insight: **xây hai bảng tần suất** `cnt1` (cho `words1`) và `cnt2` (cho `words2`), mỗi bảng chỉ cần **một lượt duyệt**.

Sau đó, một từ `w` được đếm khi và chỉ khi:

```text
cnt1[w] == 1  và  cnt2[w] == 1
```

Tra `cnt2.get(w, 0)` là trung bình `O(1)`, nên toàn bộ thuật toán là `O(n + m)` (nhân thêm độ dài từ ≤ 30 khi băm chuỗi).

In [11]:
# Thử tay trên ví dụ 1
w1 = ["leetcode", "is", "amazing", "as", "is"]
w2 = ["amazing", "leetcode", "is"]
cnt1, cnt2 = {}, {}
for w in w1:
    cnt1[w] = cnt1.get(w, 0) + 1
for w in w2:
    cnt2[w] = cnt2.get(w, 0) + 1
print("cnt1 =", cnt1)
print("cnt2 =", cnt2)

cnt1 = {'leetcode': 1, 'is': 2, 'amazing': 1, 'as': 1}
cnt2 = {'amazing': 1, 'leetcode': 1, 'is': 1}


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép lại

### 6.1. Đoạn 1: hàm đếm tần suất

Tách thành hàm riêng để dùng cho cả hai mảng (tránh lặp code).

In [12]:
def build_freq(words: list[str]) -> dict[str, int]:
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    return freq

print(build_freq(["a", "a", "a", "ab"]))  # {'a': 3, 'ab': 1}

{'a': 3, 'ab': 1}


### 6.2. Đoạn 2: duyệt bảng thứ nhất, tra bảng thứ hai

Chỉ cần duyệt `cnt1` (các từ của `words1`). Từ nào không có trong `words1` thì chắc chắn không đạt, nên không cần duyệt `cnt2`.

`cnt2.get(w, 0) == 1` xử lý luôn trường hợp `w` không có trong `words2` (trả 0 → khác 1).

In [13]:
cnt1 = build_freq(["a", "ab"])
cnt2 = build_freq(["a", "a", "a", "ab"])
for w, c1 in cnt1.items():
    c2 = cnt2.get(w, 0)
    print(f"w={w!r}: cnt1={c1}, cnt2={c2}, đếm? {c1 == 1 and c2 == 1}")

w='a': cnt1=1, cnt2=3, đếm? False
w='ab': cnt1=1, cnt2=1, đếm? True


### 6.3. Ghép lại thành hàm hoàn chỉnh

In [14]:
def count_words(words1: list[str], words2: list[str]) -> int:
    cnt1 = build_freq(words1)
    cnt2 = build_freq(words2)
    total = 0
    for w, c1 in cnt1.items():
        if c1 == 1 and cnt2.get(w, 0) == 1:
            total += 1
    return total

for w1, w2, expected in examples:
    print(count_words(w1, w2), "kỳ vọng", expected)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


### 6.4. Tại sao thứ tự điều kiện `c1 == 1 and ...` hợp lý?

`and` trong Python **đánh giá ngắn mạch** (short-circuit): nếu vế trái `False` thì không tính vế phải. Đặt `c1 == 1` trước giúp bỏ qua việc tra `cnt2` cho các từ lặp ở `words1`. Không đổi kết quả, chỉ tiết kiệm một chút.

In [15]:
def noisy(x):
    print("  đã gọi vế phải với", x)
    return True

print(False and noisy(1))  # vế phải KHÔNG được gọi
print(True and noisy(2))   # vế phải được gọi

False
  đã gọi vế phải với 2
True


## 7. TRACE từng bước — nhìn thấy trạng thái biến

### 7.1. Bảng trace cho ví dụ 1

`cnt1 = {leetcode:1, is:2, amazing:1, as:1}`, `cnt2 = {amazing:1, leetcode:1, is:1}`

| Bước | w | cnt1[w] | cnt2.get(w,0) | Đạt? | total sau |
|:---:|---|:---:|:---:|:---:|:---:|
| 1 | leetcode | 1 | 1 | ✅ | 1 |
| 2 | is | 2 | 1 | ❌ (cnt1≠1) | 1 |
| 3 | amazing | 1 | 1 | ✅ | 2 |
| 4 | as | 1 | 0 | ❌ (không có ở words2) | 2 |

Cột `cnt2.get(w,0)` cho thấy tác dụng của giá trị mặc định 0.

In [16]:
def trace_count_words(words1, words2):
    cnt1, cnt2 = {}, {}
    print("--- Pha 1: đếm words1 ---")
    for w in words1:
        cnt1[w] = cnt1.get(w, 0) + 1
        print(f"  thêm {w!r:12} -> cnt1={cnt1}")
    print("--- Pha 2: đếm words2 ---")
    for w in words2:
        cnt2[w] = cnt2.get(w, 0) + 1
        print(f"  thêm {w!r:12} -> cnt2={cnt2}")
    print("--- Pha 3: so khớp ---")
    total = 0
    for step, (w, c1) in enumerate(cnt1.items(), start=1):
        c2 = cnt2.get(w, 0)
        ok = c1 == 1 and c2 == 1
        if ok:
            total += 1
        print(f"  bước {step}: w={w!r:12} c1={c1} c2={c2} đạt={ok!s:5} total={total}")
    return total

print("Kết quả:", trace_count_words(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]))

--- Pha 1: đếm words1 ---
  thêm 'leetcode'   -> cnt1={'leetcode': 1}
  thêm 'is'         -> cnt1={'leetcode': 1, 'is': 1}
  thêm 'amazing'    -> cnt1={'leetcode': 1, 'is': 1, 'amazing': 1}
  thêm 'as'         -> cnt1={'leetcode': 1, 'is': 1, 'amazing': 1, 'as': 1}
  thêm 'is'         -> cnt1={'leetcode': 1, 'is': 2, 'amazing': 1, 'as': 1}
--- Pha 2: đếm words2 ---
  thêm 'amazing'    -> cnt2={'amazing': 1}
  thêm 'leetcode'   -> cnt2={'amazing': 1, 'leetcode': 1}
  thêm 'is'         -> cnt2={'amazing': 1, 'leetcode': 1, 'is': 1}
--- Pha 3: so khớp ---
  bước 1: w='leetcode'   c1=1 c2=1 đạt=True  total=1
  bước 2: w='is'         c1=2 c2=1 đạt=False total=1
  bước 3: w='amazing'    c1=1 c2=1 đạt=True  total=2
  bước 4: w='as'         c1=1 c2=0 đạt=False total=2
Kết quả: 2


### 7.2. Trace một nhánh đặc biệt: từ chỉ lặp ở `words2`

Ví dụ 3: `"a"` có 1 lần ở `words1` nhưng 3 lần ở `words2`. Người mới hay chỉ kiểm tra `words1` → đếm nhầm.

In [17]:
print("Kết quả:", trace_count_words(["a", "ab"], ["a", "a", "a", "ab"]))

--- Pha 1: đếm words1 ---
  thêm 'a'          -> cnt1={'a': 1}
  thêm 'ab'         -> cnt1={'a': 1, 'ab': 1}
--- Pha 2: đếm words2 ---
  thêm 'a'          -> cnt2={'a': 1}
  thêm 'a'          -> cnt2={'a': 2}
  thêm 'a'          -> cnt2={'a': 3}
  thêm 'ab'         -> cnt2={'a': 3, 'ab': 1}
--- Pha 3: so khớp ---
  bước 1: w='a'          c1=1 c2=3 đạt=False total=0
  bước 2: w='ab'         c1=1 c2=1 đạt=True  total=1
Kết quả: 1


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Brute Force với `list.count`

In [18]:
class Solution:
    def countWords(self, words1: list[str], words2: list[str]) -> int:
        total = 0
        for w in set(words1):
            if words1.count(w) == 1 and words2.count(w) == 1:
                total += 1
        return total

print([Solution().countWords(a, c) for a, c, _ in examples])

[2, 0, 1]


### Cách B — Hai dict tự đếm (lời giải chính)

In [19]:
class Solution:
    def countWords(self, words1: list[str], words2: list[str]) -> int:
        cnt1, cnt2 = {}, {}
        for w in words1:
            cnt1[w] = cnt1.get(w, 0) + 1
        for w in words2:
            cnt2[w] = cnt2.get(w, 0) + 1
        return sum(1 for w, c in cnt1.items() if c == 1 and cnt2.get(w, 0) == 1)

print([Solution().countWords(a, c) for a, c, _ in examples])

[2, 0, 1]


### Cách C — `Counter` (gọn nhất)

Vì `Counter` trả 0 cho key không có, ta viết thẳng `c2[w] == 1`.

In [20]:
from collections import Counter

class Solution:
    def countWords(self, words1: list[str], words2: list[str]) -> int:
        c1, c2 = Counter(words1), Counter(words2)
        return sum(1 for w in c1 if c1[w] == 1 and c2[w] == 1)

print([Solution().countWords(a, c) for a, c, _ in examples])

[2, 0, 1]


### Cách D — Lọc "từ xuất hiện một lần" thành set rồi giao

Bước 1: `once1 = {w : c1[w] == 1}`, `once2 = {w : c2[w] == 1}`. Bước 2: đáp án là `len(once1 & once2)`.

Cú pháp `{w for w, c in c1.items() if c == 1}` là **set comprehension**: tạo set từ các phần tử thỏa điều kiện.

In [21]:
from collections import Counter

class Solution:
    def countWords(self, words1: list[str], words2: list[str]) -> int:
        once1 = {w for w, c in Counter(words1).items() if c == 1}
        once2 = {w for w, c in Counter(words2).items() if c == 1}
        return len(once1 & once2)

print([Solution().countWords(a, c) for a, c, _ in examples])

[2, 0, 1]


### Cách E — Một dict, đánh dấu trạng thái (tiết kiệm một bảng)

Ý tưởng sáng tạo: dùng một dict `state`:
- Duyệt `words1`: lần đầu gặp → `state[w] = 1`; gặp lại → `state[w] = -1` (hỏng).
- Duyệt `words2`: chỉ từ có `state[w] == 1` lần đầu → `2`; nếu đã là `2` mà gặp lại → `-1`.
- Đáp án là số key có `state == 2`.

In [22]:
class Solution:
    def countWords(self, words1: list[str], words2: list[str]) -> int:
        state = {}
        for w in words1:
            state[w] = 1 if w not in state else -1
        for w in words2:
            s = state.get(w)
            if s == 1:
                state[w] = 2        # lần đầu ở words2
            elif s == 2:
                state[w] = -1       # lần thứ hai ở words2 -> hỏng
        return sum(1 for s in state.values() if s == 2)

print([Solution().countWords(a, c) for a, c, _ in examples])

[2, 0, 1]


## 9. Vì sao thuật toán đúng?

**Invariant sau pha đếm:** với mọi từ `w`, `cnt1[w]` (hoặc 0 nếu vắng) **bằng đúng** số lần `w` xuất hiện trong `words1`; tương tự `cnt2`. Lý do: mỗi phần tử được cộng **đúng một lần** khi duyệt.

**Không bỏ sót:** mọi từ đạt yêu cầu phải có mặt trong `words1` (vì xuất hiện 1 lần ở đó) → nó là một key của `cnt1` → được xét.

**Không đếm nhầm:** ta chỉ cộng khi cả hai số đếm bằng 1, đúng định nghĩa đề. Mỗi key của `cnt1` chỉ được duyệt một lần → không đếm trùng.

Với Cách E: trạng thái `2` chỉ đạt được khi `w` gặp đúng 1 lần ở words1 (còn `1`) rồi đúng 1 lần ở words2; mọi lần lặp thêm đều đẩy về `-1` và không thoát ra được.

## 10. Phân tích Time / Space Complexity

Gọi `n = len(words1)`, `m = len(words2)`, `L ≤ 30` là độ dài tối đa của từ (băm/so sánh chuỗi tốn `O(L)`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| A. `list.count` | `O(n·(n+m)·L)` | `O(n)` (set) | dễ nghĩ, chậm khi lớn |
| B. Hai dict | `O((n+m)·L)` | `O(n+m)` | lời giải chuẩn |
| C. `Counter` | `O((n+m)·L)` | `O(n+m)` | gọn nhất |
| D. Set giao | `O((n+m)·L)` | `O(n+m)` | đẹp, tách bước rõ |
| E. Một dict trạng thái | `O((n+m)·L)` | `O(n)` | chỉ lưu từ của words1 |

Tra cứu dict là `O(1)` **trung bình** (hash table), không phải tuyệt đối trong mọi trường hợp.

In [23]:
import random, string, time

def rand_words(k, seed):
    rng = random.Random(seed)
    return ["".join(rng.choice("ab") for _ in range(rng.randint(1, 6))) for _ in range(k)]

w1, w2 = rand_words(1000, 1), rand_words(1000, 2)
for name, f in [("brute", count_words_brute), ("dict", count_words)]:
    t = time.perf_counter(); r = f(w1, w2); dt = time.perf_counter() - t
    print(f"{name:6}: kết quả={r}, thời gian={dt*1000:.2f} ms")

brute : kết quả=5, thời gian=1.53 ms
dict  : kết quả=5, thời gian=0.16 ms


## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu nhiều cách

Nhóm test:
1. Ba ví dụ của đề.
2. Biên: mỗi mảng 1 phần tử (trùng / khác).
3. Phản ví dụ: từ đúng 1 lần ở words1 nhưng 2 lần ở words2; từ lặp ở cả hai.
4. Ngẫu nhiên có seed: đối chiếu cả 5 cách.

In [24]:
from collections import Counter

def sol_A(a, c): return count_words_brute(a, c)
def sol_B(a, c): return count_words(a, c)
def sol_C(a, c):
    c1, c2 = Counter(a), Counter(c)
    return sum(1 for w in c1 if c1[w] == 1 and c2[w] == 1)
def sol_D(a, c):
    return len({w for w, k in Counter(a).items() if k == 1} & {w for w, k in Counter(c).items() if k == 1})
def sol_E(a, c):
    state = {}
    for w in a:
        state[w] = 1 if w not in state else -1
    for w in c:
        s = state.get(w)
        if s == 1: state[w] = 2
        elif s == 2: state[w] = -1
    return sum(1 for s in state.values() if s == 2)

SOLS = [sol_A, sol_B, sol_C, sol_D, sol_E]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:35} PASS ({actual})")

tests = examples + [
    (["x"], ["x"], 1),
    (["x"], ["y"], 0),
    (["x", "x"], ["x"], 0),
    (["x"], ["x", "x"], 0),
    (["x", "y"], ["y", "x"], 2),
    (["a", "b", "a"], ["a", "b", "b"], 0),
]
for i, (a, c, exp) in enumerate(tests):
    for f in SOLS:
        check(f"test {i} {f.__name__}", f(a, c), exp)

test 0 sol_A                        PASS (2)
test 0 sol_B                        PASS (2)
test 0 sol_C                        PASS (2)
test 0 sol_D                        PASS (2)
test 0 sol_E                        PASS (2)
test 1 sol_A                        PASS (0)
test 1 sol_B                        PASS (0)
test 1 sol_C                        PASS (0)
test 1 sol_D                        PASS (0)
test 1 sol_E                        PASS (0)
test 2 sol_A                        PASS (1)
test 2 sol_B                        PASS (1)
test 2 sol_C                        PASS (1)
test 2 sol_D                        PASS (1)
test 2 sol_E                        PASS (1)
test 3 sol_A                        PASS (1)
test 3 sol_B                        PASS (1)
test 3 sol_C                        PASS (1)
test 3 sol_D                        PASS (1)
test 3 sol_E                        PASS (1)
test 4 sol_A                        PASS (0)
test 4 sol_B                        PASS (0)
test 4 sol

In [25]:
# Kiểm thử ngẫu nhiên có seed: mọi cách phải cho cùng kết quả
rng = random.Random(2085)
for trial in range(300):
    a = [rng.choice(["a", "b", "c", "d", "e"]) for _ in range(rng.randint(1, 8))]
    c = [rng.choice(["a", "b", "c", "d", "e"]) for _ in range(rng.randint(1, 8))]
    results = {f.__name__: f(a, c) for f in SOLS}
    assert len(set(results.values())) == 1, (a, c, results)
print("300 test ngẫu nhiên: 5 cách đều khớp nhau")

300 test ngẫu nhiên: 5 cách đều khớp nhau


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Chỉ kiểm tra "có mặt" ở words2 thay vì "đúng 1 lần"

In [26]:
def wrong_presence(words1, words2):
    cnt1 = build_freq(words1)
    return sum(1 for w, c in cnt1.items() if c == 1 and w in words2)   # SAI: chỉ kiểm tra có mặt

print("Sai :", wrong_presence(["a", "ab"], ["a", "a", "a", "ab"]), "(kỳ vọng 1)")
print("Đúng:", count_words(["a", "ab"], ["a", "a", "a", "ab"]))

Sai : 2 (kỳ vọng 1)
Đúng: 1


**Vì sao sai:** `w in words2` đúng cả khi `w` xuất hiện 3 lần. **Sửa:** so sánh số đếm `cnt2.get(w, 0) == 1`.

### Lỗi 2: Đọc `cnt2[w]` với dict thường → `KeyError`

In [27]:
def wrong_keyerror(words1, words2):
    cnt1, cnt2 = build_freq(words1), build_freq(words2)
    return sum(1 for w, c in cnt1.items() if c == 1 and cnt2[w] == 1)   # SAI khi w không có

try:
    wrong_keyerror(["b", "bb"], ["a"])
except KeyError as e:
    print("KeyError:", e, "-> sửa bằng cnt2.get(w, 0) hoặc dùng Counter")

KeyError: 'b' -> sửa bằng cnt2.get(w, 0) hoặc dùng Counter


### Lỗi 3: Duyệt cả hai mảng và cộng hai lần

Có người duyệt `words1` rồi lại duyệt `words2` và cộng cả hai lượt → mỗi từ đạt bị đếm **2 lần**. Chỉ cần duyệt từ phía một mảng.

In [28]:
def wrong_double(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)
    total = sum(1 for w in c1 if c1[w] == 1 and c2[w] == 1)
    total += sum(1 for w in c2 if c2[w] == 1 and c1[w] == 1)   # SAI: đếm lại
    return total

print("Sai :", wrong_double(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]), "(kỳ vọng 2)")

Sai : 4 (kỳ vọng 2)


### Lỗi 4: Dùng `set` quá sớm làm mất thông tin số lần

`len(set(words1) & set(words2))` chỉ cho biết **có chung**, đã mất số lần. Ví dụ 1 sẽ ra 3 (tính cả `"is"`). Phải lọc theo tần suất trước rồi mới giao (Cách D).

In [29]:
w1, w2 = ["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]
print("Sai :", len(set(w1) & set(w2)), "(kỳ vọng 2)")
print("Đúng:", sol_D(w1, w2))

Sai : 3 (kỳ vọng 2)
Đúng: 2


## 13. Mẹo nhận dạng pattern

Dấu hiệu nên nghĩ ngay tới **dict/Counter đếm tần suất**:

- Đề nói "xuất hiện **đúng k lần**", "**ít nhất** / **nhiều nhất** k lần", "tần suất".
- So sánh hai tập dữ liệu theo **số lần** (anagram, ransom note, giao có lặp).
- Cách brute dùng `.count()` trong vòng lặp → gần như luôn tối ưu được bằng đếm một lần.

Template:

```python
from collections import Counter
c1, c2 = Counter(A), Counter(B)
answer = sum(1 for x in c1 if điều_kiện(c1[x], c2[x]))
```

Bài liên quan: 242 Valid Anagram, 383 Ransom Note, 350 Intersection of Two Arrays II, 1207 Unique Number of Occurrences.

## 14. Biến thể — thay đổi yêu cầu để kiểm tra hiểu thật

### 14.1. Đúng `k` lần ở mỗi mảng

Chỉ thay hằng số `1` thành `k`.

In [30]:
def count_words_k(words1, words2, k):
    c1, c2 = Counter(words1), Counter(words2)
    return sum(1 for w in c1 if c1[w] == k and c2[w] == k)

print(count_words_k(["a", "a", "b", "b", "c"], ["a", "a", "b", "c", "c"], 2))  # chỉ "a"

1


### 14.2. Trả về danh sách các từ (đã sắp xếp) thay vì số lượng

In [31]:
def list_common_once(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)
    return sorted(w for w in c1 if c1[w] == 1 and c2[w] == 1)

print(list_common_once(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]))

['amazing', 'leetcode']


### 14.3. Tổng quát cho nhiều mảng

Từ phải xuất hiện đúng 1 lần ở **mọi** mảng: dùng `all(...)`.

In [32]:
def count_once_in_all(*arrays):
    counters = [Counter(a) for a in arrays]
    return sum(1 for w in counters[0] if all(c[w] == 1 for c in counters))

print(count_once_in_all(["a", "b", "c"], ["b", "c", "c"], ["b", "c"]))  # chỉ "b"

1


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Tự viết code vào các hàm TODO. Các test được để ở dạng comment; bỏ comment sau khi làm xong. Đáp án ở **mục 17**.

### Bài 1 — Đếm tần suất bằng tay
Viết `my_freq(words)` trả dict tần suất **không dùng Counter**.

In [33]:
def my_freq(words):
    # TODO: viết code ở đây
    pass

# assert my_freq(["a", "b", "a"]) == {"a": 2, "b": 1}

### Bài 2 — Số từ xuất hiện đúng 1 lần trong một mảng
`count_unique(words)`: ví dụ `["a","b","a","c"] -> 2`.

In [34]:
def count_unique(words):
    # TODO: viết code ở đây
    pass

# assert count_unique(["a", "b", "a", "c"]) == 2

### Bài 3 — Tự viết lại lời giải #2085
Viết `count_words_ex(words1, words2)` bằng dict, không nhìn lại mục 6.

In [35]:
def count_words_ex(words1, words2):
    # TODO: viết code ở đây
    pass

# assert count_words_ex(["a", "ab"], ["a", "a", "a", "ab"]) == 1

### Bài 4 — Biến thể: xuất hiện ở cả hai mảng, tổng số lần ≤ 3
`count_total_at_most3(words1, words2)`: đếm từ có mặt ở **cả hai** mảng và `cnt1 + cnt2 <= 3`.

In [36]:
def count_total_at_most3(words1, words2):
    # TODO: viết code ở đây
    pass

# assert count_total_at_most3(["a", "a", "b"], ["a", "a", "b", "c"]) == 1   # chỉ "b"

### Bài 5 — Tự tạo test có chủ đích
Viết 3 cặp `(words1, words2, expected)` mà **Lỗi 1** (mục 12) trả sai. Kiểm tra bằng `wrong_presence`.

In [37]:
my_tests = [
    # TODO: thêm (words1, words2, expected)
]
# for a, c, exp in my_tests:
#     assert count_words(a, c) == exp and wrong_presence(a, c) != exp

### Bài 6 — Mini challenge: Cách E chỉ dùng một dict
Viết lại `count_words_one_dict(words1, words2)` theo ý tưởng trạng thái `1 / 2 / -1`, không xem mục 8.

In [38]:
def count_words_one_dict(words1, words2):
    # TODO: viết code ở đây
    pass

# assert count_words_one_dict(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]) == 2

## 16. Gợi ý ba mức — chỉ mở sau khi tự thử

| Bài | Mức 1 (hướng) | Mức 2 (cấu trúc) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | Cần bảng từ → số lần | `dict` rỗng + vòng for | `d[w] = d.get(w, 0) + 1` |
| 2 | Đếm trước, lọc sau | dùng bài 1 | `sum(1 for v in d.values() if v == 1)` |
| 3 | Hai bảng tần suất | duyệt bảng 1, tra bảng 2 | `c == 1 and d2.get(w, 0) == 1` |
| 4 | "có mặt cả hai" = cả hai số > 0 | `Counter` hai mảng | `c1[w] > 0 and c2[w] > 0 and c1[w]+c2[w] <= 3` |
| 5 | Lỗi 1 sai khi words2 lặp | từ 1 lần ở words1, ≥2 lần ở words2 | `(["a"], ["a","a"], 0)` |
| 6 | Trạng thái chỉ đi một chiều | 1 → 2 → -1 | xem lại bảng trạng thái ở Cách E |

## 17. Đáp án tham khảo — đặt cuối để tránh lộ trước khi làm TODO

In [39]:
def my_freq(words):
    d = {}
    for w in words:
        d[w] = d.get(w, 0) + 1
    return d

def count_unique(words):
    return sum(1 for v in my_freq(words).values() if v == 1)

def count_words_ex(words1, words2):
    d1, d2 = my_freq(words1), my_freq(words2)
    return sum(1 for w, c in d1.items() if c == 1 and d2.get(w, 0) == 1)

def count_total_at_most3(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)
    return sum(1 for w in c1 if c2[w] > 0 and c1[w] + c2[w] <= 3)

my_tests = [
    (["a"], ["a", "a"], 0),
    (["a", "b"], ["a", "a", "b"], 1),
    (["x", "y", "z"], ["x", "x", "y", "y", "z"], 1),
]

def count_words_one_dict(words1, words2):
    state = {}
    for w in words1:
        state[w] = 1 if w not in state else -1
    for w in words2:
        if state.get(w) == 1:
            state[w] = 2
        elif state.get(w) == 2:
            state[w] = -1
    return sum(1 for s in state.values() if s == 2)

assert my_freq(["a", "b", "a"]) == {"a": 2, "b": 1}
assert count_unique(["a", "b", "a", "c"]) == 2
assert count_words_ex(["a", "ab"], ["a", "a", "a", "ab"]) == 1
assert count_total_at_most3(["a", "a", "b"], ["a", "a", "b", "c"]) == 1
for a, c, exp in my_tests:
    assert count_words(a, c) == exp and wrong_presence(a, c) != exp
assert count_words_one_dict(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]) == 2
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1–2: đếm một lượt, rồi lọc theo giá trị.
- Bài 3: đúng lời giải chính; điểm mấu chốt là `get(w, 0)`.
- Bài 4: "có mặt ở cả hai" nghĩa là `c2[w] > 0` (và `c1[w] > 0` đã đúng vì duyệt `c1`).
- Bài 5: Lỗi 1 chỉ sai khi từ xuất hiện ≥ 2 lần ở `words2` mà đúng 1 lần ở `words1`.
- Bài 6: trạng thái chỉ tiến (1 → 2) hoặc hỏng (→ -1), không bao giờ quay lại.

## 18. Tổng kết — template suy nghĩ + checklist nộp bài

**Template suy nghĩ cho bài "đếm theo số lần":**
1. Đề hỏi về **số lần** → cần bảng tần suất.
2. Đếm mỗi mảng **một lần** (`Counter` hoặc `dict.get`).
3. Duyệt bảng nhỏ / bảng chứa ứng viên, tra bảng còn lại.
4. Cẩn thận key vắng mặt (`get(w, 0)` hoặc `Counter`).

**Checklist trước khi submit:**
- [ ] Kiểm tra **cả hai** điều kiện `== 1`, không chỉ "có mặt".
- [ ] Không `KeyError` khi từ không có ở mảng thứ hai.
- [ ] Không đếm một từ hai lần.
- [ ] Đúng tên method `countWords` và trả về `int`.
- [ ] Chạy lại 3 ví dụ của đề: `2, 0, 1`.